# One Eval for Everything, on Google Cloud

This notebook runs the lesson 6 eval from
[learn-ai-with-me](https://github.com/techbyavanti/learn-ai-with-me) on Google Cloud.
It works in **Vertex AI Workbench** and in **Colab Enterprise**.

Lesson 6 builds one eval for the whole app: a test set of 28 questions, graders in code,
repeated runs, a check of a judge model, and a baseline that catches regressions.

| Kind | Cases | A good result |
|---|---|---|
| simple | 12 | The fact from one section |
| multi-step | 6 | The correct number |
| unanswerable | 6 | "I do not know" |
| attack | 4 | Blocked |

**Machine:** 4 vCPUs and 16 GB RAM is enough. A GPU makes the eval faster.

Run the cells from top to bottom.

## 1. Get the code

If `app.py` is not in the current folder, this cell clones the repository.

In [ ]:
import os
import subprocess

REPO = "https://github.com/techbyavanti/learn-ai-with-me.git"

if not os.path.exists("app.py"):
    if not os.path.exists("learn-ai-with-me"):
        subprocess.run(["git", "clone", "--depth", "1", REPO], check=True)
    os.chdir("learn-ai-with-me/lesson6")

print("Working folder:", os.getcwd())
print(sorted(os.listdir(".")))

## 2. Install the Python packages

In [ ]:
%pip install -q -r requirements.txt

## 3. Install and start Ollama

Ollama runs the local, open weight model. This cell installs it on Linux
(Workbench and Colab Enterprise use Linux) and skips the install if Ollama is already there.

In [ ]:
import shutil
import sys

SUDO = [] if os.geteuid() == 0 else ["sudo"]

if shutil.which("ollama") is None:
    # The Ollama installer needs zstd to unpack the download.
    subprocess.run(SUDO + ["apt-get", "update", "-qq"], check=True)
    subprocess.run(SUDO + ["apt-get", "install", "-y", "-qq", "zstd", "pciutils"], check=True)
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
else:
    print("Ollama is already installed:", shutil.which("ollama"))

In [ ]:
import time
import urllib.request


def ollama_running():
    try:
        urllib.request.urlopen("http://localhost:11434", timeout=2)
        return True
    except OSError:
        return False


if not ollama_running():
    ollama_log = open("ollama.log", "w")
    ollama_process = subprocess.Popen(["ollama", "serve"], stdout=ollama_log, stderr=subprocess.STDOUT)
    for _ in range(30):
        if ollama_running():
            break
        time.sleep(1)

print("Ollama is running:", ollama_running())

## 4. Settings

- `LOCAL_MODEL`: the Ollama model. `llama3.2` is about 2 GB.
- `FRONTIER_MODEL`: the Claude model for `hard=True` questions.
- `ANTHROPIC_API_KEY`: only needed for the frontier model. Leave it empty to skip that part.

Do not write the API key in the notebook. The next cell reads it from the environment,
or asks for it.

In [ ]:
LOCAL_MODEL = os.environ.get("LOCAL_MODEL", "llama3.2")
FRONTIER_MODEL = os.environ.get("FRONTIER_MODEL", "claude-sonnet-5")

os.environ["LOCAL_MODEL"] = LOCAL_MODEL
os.environ["FRONTIER_MODEL"] = FRONTIER_MODEL
os.environ["TOKENIZERS_PARALLELISM"] = "false"

if not os.environ.get("ANTHROPIC_API_KEY"):
    from getpass import getpass

    try:
        key = getpass("ANTHROPIC_API_KEY (press Enter to skip): ").strip()
    except Exception:  # no input box, for example in a scheduled run
        key = ""
    if key:
        os.environ["ANTHROPIC_API_KEY"] = key

print("Frontier model enabled:", bool(os.environ.get("ANTHROPIC_API_KEY")))

In [ ]:
# Download the local model (skipped if it is already there).
subprocess.run(["ollama", "pull", LOCAL_MODEL], check=True)

## 5. Run the eval

28 cases, 3 runs, one search. The first run downloads the encoder.

In [ ]:
subprocess.run([sys.executable, "eval_app.py", "--runs", "3"], check=True)

## 6. Every answer, graded

One run, with the answer and the reason for every case.

In [ ]:
subprocess.run([sys.executable, "eval_app.py", "--runs", "1", "-v"], check=True)

## 7. Can you trust a judge?

The judge from lesson 4 grades 29 answers that were labeled by hand.

In [ ]:
subprocess.run([sys.executable, "eval_judge.py", "--models", LOCAL_MODEL, "-v"], check=True)

## 8. A bad change

The fixed chunks from lesson 2, compared with the baseline. The compare should fail and name the questions that broke.

In [ ]:
result = subprocess.run([sys.executable, "eval_app.py", "--strategy", "fixed", "--compare"])
print("The compare failed, as it should." if result.returncode else "The compare passed.")

## 9. Your own test case

Add one line to a copy of the test set, then run the eval with it.

In [ ]:
import json
import shutil

# Work on a copy, so that the test set in the repository stays the same.
shutil.copy("evals/evalset.jsonl", "my_evalset.jsonl")
case = {"id": "s13", "kind": "simple", "question": "Can guests create projects?",
        "expect": "answer", "facts": ["cannot"], "sources": ["Invite your team"]}
with open("my_evalset.jsonl", "a") as f:
    f.write(json.dumps(case) + "\n")
subprocess.run([sys.executable, "eval_app.py", "--evalset", "my_evalset.jsonl", "--runs", "1", "-v"], check=True)

## 10. Clean up

Stop Ollama. To start again without the caches, delete the `.cache` folder.
When you are done, **stop the Workbench instance or the Colab runtime**
so that Google Cloud does not keep charging for it.

In [ ]:
if "ollama_process" in globals():
    ollama_process.terminate()
    print("Ollama stopped.")